In [3]:
import numpy as np
import pandas as pd
import tensorflow as tf
from gensim.models import Word2Vec
from gensim.utils import simple_preprocess, tokenize
from tensorflow import keras
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from keras.models import Sequential
from keras.layers import Embedding, LSTM, Dense
from keras.utils import to_categorical

In [4]:
sample_messages_file_path = r"C:\Users\Lenovo\Desktop\Hot Hand\Programming\Programming docs\Datasets\2col-nlp-classification-comments-bool-dataset.csv"
samples = pd.read_csv(sample_messages_file_path)
samples = samples[samples.columns[0]].to_list()

In [5]:
tokenized_samples = [simple_preprocess(line) for line in samples]

In [6]:
w2v_model = Word2Vec(sentences=tokenized_samples, vector_size=100, min_count=2, window=5, workers=4)

In [7]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts(samples)
word_to_index = tokenizer.word_index  
vocab_size = len(word_to_index) + 1
embedding_dim = 100

In [8]:
embedding_matrix = np.zeros((vocab_size, embedding_dim))
for word, idx in word_to_index.items():
    if word in w2v_model.wv: embedding_matrix[idx] = w2v_model.wv[word]


In [9]:
input_sequences = []
for line in samples:
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(token_list)):
        n_gram_sequence = token_list[:i+1]
        input_sequences.append(n_gram_sequence)

In [10]:
max_len = max(len(seq) for seq in input_sequences)
input_sequences = pad_sequences(input_sequences, maxlen=max_len, padding="pre")
X = input_sequences[:, :-1]         
y = input_sequences[:, -1]          
y = to_categorical(y, num_classes=vocab_size)

model = Sequential([
    Embedding(
        input_dim=vocab_size,
        output_dim=embedding_dim,
        weights=[embedding_matrix],
        trainable=True,
        input_length=max_len - 1
    ),
    LSTM(86),
    Dense(vocab_size, activation="softmax")
])

c:\Users\Lenovo\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\core\embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [11]:
model.compile(loss="categorical_crossentropy", optimizer="adam", metrics=["accuracy"])

In [12]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │       206,900 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 206,900 (808.20 KB)

 Trainable params: 206,900 (808.20 KB)

 Non-trainable params: 0 (0.00 B)

In [14]:
history = model.fit(X, y, epochs=100, batch_size=32, verbose=1)

Epoch 1/100
309/309 ━━━━━━━━━━━━━━━━━━━━ 14s 39ms/step - accuracy: 0.7485 - loss: 1.1827
Epoch 2/100
309/309 ━━━━━━━━━━━━━━━━━━━━ 12s 36ms/step - accuracy: 0.7537 - loss: 1.1509
Epoch 3/100
309/309 ━━━━━━━━━━━━━━━━━━━━ 9s 29ms/step - accuracy: 0.7589 - loss: 1.1204
Epoch 4/100
309/309 ━━━━━━━━━━━━━━━━━━━━ 8s 26ms/step - accuracy: 0.7668 - loss: 1.0891
Epoch 5/100
309/309 ━━━━━━━━━━━━━━━━━━━━ 8s 25ms/step - accuracy: 0.7721 - loss: 1.0603
Epoch 6/100
309/309 ━━━━━━━━━━━━━━━━━━━━ 9s 28ms/step - accuracy: 0.7792 - loss: 1.0309
Epoch 7/100
309/309 ━━━━━━━━━━━━━━━━━━━━ 7s 23ms/step - accuracy: 0.7852 - loss: 1.0018
Epoch 8/100
309/309 ━━━━━━━━━━━━━━━━━━━━ 12s 29ms/step - accuracy: 0.7935 - loss: 0.9763
Epoch 9/100
309/309 ━━━━━━━━━━━━━━━━━━━━ 8s 25ms/step - accuracy: 0.7942 - loss: 0.9509
Epoch 10/100
309/309 ━━━━━━━━━━━━━━━━━━━━ 7s 23ms/step - accuracy: 0.8018 - loss: 0.9287
Epoch 11/100
309/309 ━━━━━━━━━━━━━━━━━━━━ 7s 22ms/step - accuracy: 0.8105 - loss: 0.9025
Epoch 12/100
309/309 ━━━━━━

In [13]:

def predict_next_word(seed_text, num_words=1):
    for _ in range(num_words):
        token_list = tokenizer.texts_to_sequences([seed_text])[0]
        token_list = pad_sequences([token_list], maxlen=max_len - 1, padding="pre")
        predicted_probs = model.predict(token_list, verbose=0)
        predicted_index = np.argmax(predicted_probs, axis=-1)[0]
        output_word = ""
        
        for word, index in word_to_index.items():
            if index == predicted_index:
                output_word = word
                break
        seed_text += " " + output_word
    return seed_text

In [26]:
if __name__ == "__main__" : print(predict_next_word("It is a", num_words=1))

It is a particular
